# Optional extension — Circular motion — Newton’s laws applied to curves

## Dairesel hareket

**Role in the course:** Supports Week 10 (rotation). Centripetal acceleration, banked curves, loops and the conical pendulum.

This notebook is **not** a scheduled calendar week. Use it for deeper reading, extra examples and practice after the related weekly notebook. Its problems keep the identifier *Module 05 Pn* for the solution collection.

**TR:** Bu not takvimde ayrı bir hafta değildir; ilgili haftadan sonra ek okuma ve alıştırma için kullanılır.

## Contents / İçindekiler

1. [Before you start](#x05-before)
   · [Setup for the interactive graphs (run once)](#x05-setup)
2. [Concepts, demonstrations and worked examples](#x05-concepts) — 3 worked examples, 4 interactive graphs
3. [Problem set — predict, then check](#x05-problems) — 10 problems

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)


<a id="x05-before"></a>

## 1. Before you start / Başlamadan önce

### Learning Objectives

By the end of this session you will be able to:

1. Derive and apply the centripetal acceleration formula $a_c = \frac{v^2}{r}$
2. Analyze forces on an object moving in a horizontal or vertical circle
3. Solve banked-curve problems (with and without friction)
4. Determine the minimum speed at the top of a vertical loop (loop-the-loop)
5. Analyze a conical pendulum and relate the cone angle to angular speed
6. Build intuition through interactive simulations of circular-motion scenarios

### Before calculating: use Newton's law toward the centre

**Core route:** draw the circle centre → choose inward positive → identify real forces → write $\sum F_{\rm inward}=m\frac{v^2}{r}$ → solve by hand → use the plot as a check. Coding a simulation is optional.

**What is new?** A constant speed can still mean changing velocity because direction changes. The inward acceleration is $a_r=\frac{v^2}{r}$. Centripetal force is the **sum of actual radial forces**, not a separate force to add to gravity or tension.

**Algebra bridge:** on a flat curve, $\mu_smg=m\frac{v^2}{r}$ at the grip limit. Divide both sides by nonzero $m$, multiply by $r$, then take the positive square root for speed:
$$\mu_sg=\frac{v^2}{r}\ \Rightarrow\ \mu_sgr=v^2\ \Rightarrow\ v=\sqrt{\mu_sgr}.$$
Inside the root, $(\mathrm{m/s^2})(\mathrm m)=\mathrm{m^2/s^2}$. A common error is taking $v=\mu_sgr$ and forgetting the square root.

**Direction check:** at the top of an inside loop, inward is down, so $mg+N=m\frac{v^2}{r}$. At the bottom, inward is up, so $N-mg=m\frac{v^2}{r}$. For a car on top of an outside hill, inward is down but $N$ points up: $mg-N=m\frac{v^2}{r}$.

**Notation:** tension and period are sometimes both called $T$. Label them $T_{\rm tension}$ and $T_{\rm period}$ in your own work. A conical pendulum's path radius is $r=L\sin\theta$, not $L$.

**Preview and buffer:** loop problems that relate speeds at different heights use energy, developed in the energy notes. First practise radial force balance with a given speed. The friction-loop challenge additionally needs an integral because $N$ changes; its advanced calculation is not a programming requirement.

**Review stop:** if speed doubles at the same radius, acceleration quadruples. If radius doubles at the same speed, acceleration halves. Predict both before moving a control.

**Türkçe:** Merkezcil kuvvet yeni bir kuvvet değildir. Merkeze doğru olan gerçek kuvvetlerin toplamıdır. Merkez yönünü her noktada yeniden göster.

<a id="x05-setup"></a>

## Setup for the interactive graphs (run once) / Kurulum — bir kez çalıştır

Run the cells in this section once per session, then run any **Run the demonstration** cell below. Each demonstration shows a status line under its controls: **Updating…** while the graph is drawn, then the draw time and whether the sliders update live or on release. Nothing here needs to be edited.  
**TR:** Bu bölümdeki hücreleri oturum başına bir kez çalıştır; sonra istediğin gösterimi çalıştır. Kontrollerin altındaki durum satırı, grafiğin ne zaman güncellendiğini gösterir.

In [ ]:
#@title Run once — prepare the physics demonstrations
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, HBox, VBox
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (8, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print('All imports successful.')

In [ ]:
#@title Run once — prepare the demonstration controls
"""Shared demonstration interface, embedded in every PHY101 notebook.

Only standard ipywidgets, IPython and matplotlib are used, so the notebooks stay
self-contained in Colab and in a local Jupyter. The design goals are:

* A slider change must always produce a visible reaction. The status line under
  the controls says "Updating…" immediately and reports the draw time afterwards.
* The graph is replaced through the same Output-widget route that
  ``ipywidgets.interact`` uses (``clear_output(wait=True)`` followed by a fresh
  display), which is the most widely tested path in Colab and Jupyter. No
  output-capturing context is used: ipykernel 7 dispatches widget messages
  concurrently and IPython's capture object breaks that dispatch.
* Live updates while dragging are switched on when a graph draws quickly and
  switched off (update on release) when it draws slowly, so the kernel never
  falls behind a fast slider.
"""
import functools
import sys
import time
import traceback

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import HTML, clear_output, display

# Force the inline backend. Otherwise a local kernel may choose a desktop
# backend and block at plt.show(), which looks like a frozen notebook.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

_physics_panels = []
_physics_callback_errors = []

# Draw-time thresholds (seconds) for switching live dragging on and off.
PHYSICS_LIVE_ON = 0.12
PHYSICS_LIVE_OFF = 0.25


def physics_frames(frame_count, maximum=60):
    """Sample display frames, retaining both endpoints and all simulation data."""
    count = int(frame_count)
    shown = min(count, maximum)
    if shown <= 1:
        return list(range(shown))
    return [round(index * (count - 1) / (shown - 1)) for index in range(shown)]


def physics_interval(frame_count, interval_ms):
    """Preserve first-to-last playback duration when display frames are sampled."""
    shown = len(physics_frames(frame_count))
    return interval_ms if shown <= 1 else interval_ms * (int(frame_count) - 1) / (shown - 1)


PHYSICS_STYLE = """<style>
.phy101-panel { border: 1px solid #a9b9c9; border-radius: 8px; padding: 8px; background: #fff; }
.phy101-controls { padding: 0 0 2px; box-sizing: border-box; }
.phy101-status { font-size: 12px; color: #4a5a6a; padding: 0 2px 6px; min-height: 18px; }
.phy101-status.busy { color: #b45309; }
.phy101-plot-output img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-plot-output .output_area { overflow: visible; }
.phy101-plot-output table { font-size: 13px; width: 100%; }
.phy101-plot-output .animation { max-width: 100%; }
.phy101-plot-output .animation img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-animation-panel { max-width: 100%; }
.phy101-animation-panel .animation { display: flex; flex-direction: column; }
.phy101-animation-panel .animation img { order: 2; max-width: 100%; height: auto; }
.phy101-animation-panel .anim-controls { order: 1; background: white; color: #172433; padding: 4px; }
@media (max-width: 650px) {
  .phy101-controls, .phy101-plot-output { width: 100% !important; }
}
</style>"""
display(HTML(PHYSICS_STYLE))


def physics_animation_html(animation):
    """A self-contained animation pane with playback controls kept in view."""
    return HTML(PHYSICS_STYLE + '<div class="phy101-animation-panel">' +
                animation.to_jshtml(default_mode="once") + "</div>")


def _physics_controls(items):
    """Lay the controls out as a wrapping toolbar with full-length labels."""
    flat = []
    for item in items:
        if isinstance(item, (widgets.HBox, widgets.VBox)):
            flat.extend(item.children)
        else:
            flat.append(item)
    for control in flat:
        if hasattr(control, "style") and "description_width" in control.style.traits():
            control.style.description_width = "initial"
        control.layout.width = "310px"
        control.layout.flex = "0 1 310px"
        control.layout.max_width = "100%"
        control.layout.min_width = "0"
        control.layout.margin = "2px 6px 2px 0"
        if isinstance(control, widgets.Button):
            control.layout.width = "auto"
            control.layout.flex = "0 0 auto"
    # Repeat the style inside the widget tree: Colab isolates output frames.
    style = widgets.HTML(value=PHYSICS_STYLE, layout=widgets.Layout(display="none"))
    box = widgets.Box([style] + flat, layout=widgets.Layout(
        display="flex", flex_flow="row wrap", align_items="center",
        width="100%", min_width="0", max_width="100%"))
    box.add_class("phy101-controls")
    return box


def _physics_output(output):
    output.layout = widgets.Layout(
        width="100%", min_width="0", max_width="100%",
        height="auto", overflow="visible", margin="0")
    output.add_class("phy101-plot-output")
    return output


def physics_panel(controls, output):
    """Button-driven demos: a compact control toolbar directly above the result."""
    panel = widgets.Box([_physics_controls(controls), _physics_output(output)],
        layout=widgets.Layout(display="flex", flex_flow="column",
                              align_items="stretch", width="100%"))
    panel.add_class("phy101-panel")
    return panel


def physics_show_figure(figure):
    """Display one inline figure and close its pyplot registration afterwards."""
    import matplotlib.pyplot as plt
    display(figure)
    plt.close(figure)


def physics_vector_axes(axes, points):
    """Equal x/y scales and limits covering all arrow endpoints, including sums."""
    import numpy as np
    coordinates = np.asarray(points, dtype=float).reshape(-1, 2)
    span = max(1.0, float(np.max(np.abs(coordinates)))) * 1.22
    axes.set(xlim=(-span, span), ylim=(-span, span), xlabel="x component", ylabel="y component")
    axes.set_aspect("equal", adjustable="box")
    axes.axhline(0, color="#718096", linewidth=0.7)
    axes.axvline(0, color="#718096", linewidth=0.7)
    axes.grid(alpha=0.2)


class PhysicsPanel:
    """Controls, a status line and one Output widget that shows the latest result."""

    def __init__(self, function, controls):
        self.f = function
        self.controls = controls
        self.out = _physics_output(widgets.Output())
        self.status = widgets.HTML(value="")
        self.status.add_class("phy101-status")
        self.seconds = None
        self.live = True
        self.updates = 0
        self.last_outputs = 0
        self.figures = 0
        self.error = None
        visible = []
        for control in controls.values():
            if isinstance(control, widgets.fixed):
                continue
            visible.append(control)
            if hasattr(control, "continuous_update"):
                control.continuous_update = True
            control.observe(self._changed, names="value")
        self.widget = widgets.VBox([_physics_controls(visible), self.status, self.out],
                                   layout=widgets.Layout(width="100%"))
        self.widget.add_class("phy101-panel")
        self.children = self.widget.children
        _physics_panels.append(self)
        self.render()

    # Compatibility with the earlier validation code.
    @property
    def layout(self):
        return self.widget.layout

    def _changed(self, change):
        self.render()

    def _set_live(self, live):
        if live == self.live:
            return
        self.live = live
        for control in self.controls.values():
            if hasattr(control, "continuous_update"):
                control.continuous_update = live

    def render(self):
        self.status.value = "⏳ Updating… / Güncelleniyor…"
        self.status.add_class("busy")
        started = time.perf_counter()
        kwargs = {name: control.value for name, control in self.controls.items()}
        self.error = None
        self.figures = 0
        # Count everything the demonstration shows (figures, HTML, animations, text)
        # by wrapping the display publisher and stdout for this draw only.
        shell = get_ipython()
        publisher = getattr(shell, "display_pub", None) if shell is not None else None
        if publisher is not None:
            original_publish = publisher.publish

            def counting_publish(*args, **kwargs):
                self.figures += 1
                return original_publish(*args, **kwargs)
            publisher.publish = counting_publish
        stdout = sys.stdout
        original_write = stdout.write

        def counting_write(text):
            if text.strip():
                self.figures += 1
            return original_write(text)
        stdout.write = counting_write
        try:
            # The previous result stays visible until the new one arrives.
            with self.out:
                clear_output(wait=True)
                try:
                    result = self.f(**kwargs)
                    from ipywidgets.widgets.interaction import show_inline_matplotlib_plots
                    show_inline_matplotlib_plots()
                    if result is not None:
                        display(result)
                except Exception:
                    self.error = traceback.format_exc()
                    _physics_callback_errors.append((getattr(self.f, "__name__", "callback"), self.error))
                    print(self.error)
        finally:
            if publisher is not None and publisher.__dict__.get("publish") is counting_publish:
                del publisher.publish
            if stdout.__dict__.get("write") is counting_write:
                del stdout.write
        self.last_outputs = self.figures
        self.seconds = time.perf_counter() - started
        self.updates += 1
        if self.seconds > PHYSICS_LIVE_OFF:
            self._set_live(False)
        elif self.seconds < PHYSICS_LIVE_ON:
            self._set_live(True)
        self.status.remove_class("busy")
        mode = ("updates while you drag / sürüklerken güncellenir" if self.live
                else "updates when you release the slider / kaydırıcıyı bırakınca güncellenir")
        self.status.value = (f"✓ Drawn in {self.seconds:.2f} s · {mode}" if self.error is None
                             else "⚠ The demonstration reported an error; see the message below.")


def physics_interactive(function, **controls):
    """Build a panel like ipywidgets.interactive, returning the panel object."""
    @functools.wraps(function)
    def checked(*args, **kwargs):
        return function(*args, **kwargs)

    return PhysicsPanel(checked, controls)


def physics_interact(function=None, **controls):
    """Support both @physics_interact(...) and physics_interact(function, ...)."""
    if function is None:
        return lambda function: physics_interact(function, **controls)
    panel = physics_interactive(function, **controls)
    function.widget = panel.widget
    function.panel = panel
    display(panel.widget)
    return function


<a id="x05-concepts"></a>

## 2. Concepts, demonstrations and worked examples / Konular, gösterimler ve çözümlü örnekler

### Meaning and geometry: acceleration can turn a velocity arrow

**Optional enrichment; no additional common-exam scope.** Place two equal-length velocity arrows at nearby points on a circular path. Translate their tails together and draw the difference arrow. Its direction tends inward as the time interval shrinks. Thus constant speed still permits acceleration: the vector velocity changes direction.

**Predict.** Double the speed at fixed radius. Does the inward acceleration double? Which actual interaction supplies it for a car on a level bend?

<details><summary>Read the geometry and name the agent</summary>

It quadruples because $a_r=v^2/r$. One factor of speed comes from the velocity-arrow length and another from how quickly its direction turns. For a car on a level bend, static tyre–road friction supplies the horizontal inward force within the no-slip model. Do not draw an additional “centripetal force” beside it.

</details>

A robot end effector moving around an arc and a parcel constrained in a rotating drum obey the same vector-change idea, but the agents supplying their radial forces differ.

**TR:** Sürat sabitken hız vektörünün yönü değişebilir; ivmeyi sağlayan gerçek etkileşimi adlandır.

### Theory: Uniform Circular Motion

#### Why does an object moving in a circle accelerate?

Velocity includes direction. Even when speed $v$ is constant, turning changes the velocity vector, so the object accelerates toward the centre. This section assumes a fixed radius $r>0$ and uniform speed.

<table width="100%">
<thead>
<tr>
<th align="left" width="176" scope="col">Quantity</th>
<th align="left" width="88" scope="col">Symbol</th>
<th align="left" width="152" scope="col">Formula</th>
<th align="left" width="80" scope="col">SI unit</th>
</tr>
</thead>
<tbody>
<tr>
<td>Period</td>
<td>$T_{\rm period}$</td>
<td>$\displaystyle T_{\rm period}=\frac{2\pi r}{v}$</td>
<td>$\mathrm s$</td>
</tr>
<tr>
<td>Frequency</td>
<td>$f$</td>
<td>$f=\frac{1}{T_{\rm period}}$</td>
<td>$\mathrm{Hz}=\mathrm{s^{-1}}$</td>
</tr>
<tr>
<td>Angular speed</td>
<td>$\omega$</td>
<td>$\omega=2\pi f=\frac vr$</td>
<td>$\mathrm{rad/s}$</td>
</tr>
<tr>
<td>Radial acceleration</td>
<td>$a_c$</td>
<td>$a_c=\frac{v^2}{r}=\omega^2r$</td>
<td>$\mathrm{m/s^2}$</td>
</tr>
<tr>
<td>Net inward force</td>
<td>$\sum F_r$</td>
<td>$\sum F_r=ma_c=m\frac{v^2}{r}$</td>
<td>$\mathrm N$</td>
</tr>
</tbody>
</table>

**Small example:** A $0.50\,\mathrm{kg}$ object moves at $2.0\,\mathrm{m/s}$ on a circle of radius $1.0\,\mathrm m$.

$$a_c=\frac{(2.0\,\mathrm{m/s})^2}{1.0\,\mathrm m}=4.0\,\mathrm{m/s^2},\qquad \sum F_r=(0.50)(4.0)=2.0\,\mathrm N.$$

At $4.0\,\mathrm{m/s}$ on the same circle, $a_c=16\,\mathrm{m/s^2}$ and inward force is $8.0\,\mathrm N$: both quadruple.

**Türkçe:** Sürat sabit kalırken yön değişebilir. İvme yalnızca süratin değişmesi değildir. Hızın karesini önce al; iki kat hız dört kat merkezcil ivme gerektirir.

#### Key idea

**Centripetal force is not an additional force.** It names the net radial contribution of gravity, friction, tension and/or normal force. Draw those interactions, then add their inward components. If speed also changes, a tangential acceleration exists in addition to the radial component.

#### Analogy -- The hammer throw

Before release, cable tension helps turn the hammer. Its instantaneous velocity is tangent to the circle. Immediately after release it starts along that tangent; gravity then curves the flight into a projectile path. A straight path forever would require neglecting gravity too.

### Interactive: Circular Motion with Vectors

Watch an object travel around a circle. The **blue arrow** is the velocity (tangent) and the **red arrow** is the centripetal acceleration (toward centre). Use the slider to change the speed.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def circular_motion_animation(speed=2.0):
    """Animate an object in uniform circular motion showing v and a vectors."""
    R = 3.0  # radius
    omega = speed / R
    T = 2 * np.pi / omega if omega != 0 else 1e6
    dt = 0.05
    n_frames = int(min(T / dt, 200))

    fig, ax = plt.subplots(figsize=(7, 7))
    extent = max(4.5, np.hypot(R, 0.8*speed)+1.0)
    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_aspect('equal')
    ax.set_title(f'Uniform Circular Motion  (v = {speed:.1f} m/s, R = {R:.1f} m)', fontsize=13)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')

    # Draw circle path
    theta_path = np.linspace(0, 2*np.pi, 200)
    ax.plot(R*np.cos(theta_path), R*np.sin(theta_path), 'k--', alpha=0.3, lw=1)
    ax.plot(0, 0, 'k+', markersize=10)  # centre

    ball, = ax.plot([], [], 'ko', markersize=12)
    v_arrow = ax.annotate('', xy=(0,0), xytext=(0,0),
                          arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    a_arrow = ax.annotate('', xy=(0,0), xytext=(0,0),
                          arrowprops=dict(arrowstyle='->', color='red', lw=2))
    v_label = ax.text(0, 0, '', color='blue', fontsize=11, fontweight='bold')
    a_label = ax.text(0, 0, '', color='red', fontsize=11, fontweight='bold')
    info_text = fig.text(0.10, 0.04, '', fontsize=10, verticalalignment='bottom',
                        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    def init():
        ball.set_data([], [])
        return ball,

    def update(frame):
        t = frame * dt
        theta = omega * t
        x = R * np.cos(theta)
        y = R * np.sin(theta)
        ball.set_data([x], [y])

        # Velocity: tangent direction
        vx = -speed * np.sin(theta)
        vy =  speed * np.cos(theta)
        scale_v = 0.8
        v_arrow.xy = (x + scale_v*vx, y + scale_v*vy)
        v_arrow.set_position((x, y))
        v_label.set_position((x + scale_v*vx*0.5 - 0.6, y + scale_v*vy*0.5 + 0.2))
        v_label.set_text('v')

        # Acceleration: toward centre
        ac = speed**2 / R
        ax_dir = -np.cos(theta)  # unit vector toward centre
        ay_dir = -np.sin(theta)
        scale_a = 0.3
        a_arrow.xy = (x + scale_a*ac*ax_dir, y + scale_a*ac*ay_dir)
        a_arrow.set_position((x, y))
        a_label.set_position((x + scale_a*ac*ax_dir*0.5 + 0.2, y + scale_a*ac*ay_dir*0.5 - 0.3))
        a_label.set_text('$a_c$')

        info_text.set_text(
            f'$\\omega$ = {omega:.2f} rad/s\n'
            f'T = {T:.2f} s\n'
            f'$a_c$ = v$^2$/R = {ac:.2f} m/s$^2$'
        )
        return ball,

    fig.tight_layout(rect=[0, 0.20, 1, 1])
    anim = FuncAnimation(fig, update, init_func=init,
                         frames=physics_frames(n_frames), interval=physics_interval(n_frames, 50), blit=False)
    plt.close(fig)
    return physics_animation_html(anim)

physics_interact(circular_motion_animation,
         speed=FloatSlider(min=0.5, max=6.0, step=0.5, value=2.0,
                           description='Speed (m/s)'));

#### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

**Think:** A puck moves at $2\,\mathrm{m/s}$ on a $1\,\mathrm m$ radius circle. Compare inward acceleration when (a) speed doubles and (b) radius doubles at the original speed.

**Pair:** Compare the chosen signs and the equation before calculating.

**Explain:** Give the result, its unit, and one sentence of physical meaning.

**Türkçe:** Önce tek başına tahmin et; sonra arkadaşınla eksen ve denklem seçimini karşılaştır. Aşağıdaki model açıklamayı kendi gerekçenden sonra oku.

<details><summary>Model answer / Örnek yanıt — open after trying</summary>

Keep the unchanged variable explicit in each comparison:

$$\begin{aligned}
a_{c,0}&=\frac{(2\,\mathrm{m/s})^2}{1\,\mathrm m}=4\,\mathrm{m/s^2},\\
a_{c,\,2v}&=\frac{(4\,\mathrm{m/s})^2}{1\,\mathrm m}=16\,\mathrm{m/s^2},\\
a_{c,\,2r}&=\frac{(2\,\mathrm{m/s})^2}{2\,\mathrm m}=2\,\mathrm{m/s^2}.
\end{aligned}$$

Doubling speed quadruples acceleration; doubling radius at the original speed halves it. **Türkçe:** Hız payda değil, paydaki kareli terimdedir. Her satırda hangi değişkenin sabit kaldığını kontrol et.

</details>

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### Theory: Banked Curves

A banked road tilts the normal force, allowing it to contribute inward force. Use a rear-view diagram with inward horizontal and upward vertical axes.

#### Frictionless banked curve

For a bank angle $\theta$ and turn radius $r$, no vertical acceleration gives $N\cos\theta=mg$. Horizontal circular motion gives $N\sin\theta=m\frac{v^2}{r}$. Divide the **entire** second equation by the first:

$$
\begin{aligned}
\frac{N\sin\theta}{N\cos\theta}&=\frac{m\frac{v^2}{r}}{mg},\\
\tan\theta&=\frac{v^2}{rg},\\
v^2&=rg\tan\theta,\\
v_{\rm ideal}&=\sqrt{rg\tan\theta}.
\end{aligned}
$$

The positive root is speed. Both $N$ and $m$ cancel; the bank and radius determine one no-friction speed.

**Türkçe:** İki denklemde aynı normal kuvvet ve aynı kütle bulunduğu için bölmeyle bilinmeyenleri yok ediyoruz. Son satırda hâlâ $v^2$ varsa hız bulunmuş değildir; karekök gerekir.

#### With friction

<table width="100%">
<thead>
<tr>
<th align="left" width="208" scope="col">Speed relative to ideal</th>
<th align="left" width="520" scope="col">Friction needed to prevent slipping</th>
</tr>
</thead>
<tbody>
<tr>
<td>$v<v_{\rm ideal}$</td>
<td>Up the bank; prevents sliding toward the lower/inside side</td>
</tr>
<tr>
<td>$v=v_{\rm ideal}$</td>
<td>No friction needed</td>
</tr>
<tr>
<td>$v>v_{\rm ideal}$</td>
<td>Down the bank; prevents sliding toward the higher/outside side</td>
</tr>
</tbody>
</table>

**Small limit check:** Let $\tan\theta=0.20$ and $\mu_s=0.30$. The expression for the lower speed limit has numerator $\tan\theta-\mu_s=-0.10$. This does not require an imaginary speed; friction is sufficient even at rest, so the physical lower limit is zero. When $1-\mu_s\tan\theta\le0$, this ideal model supplies no finite upper speed bound; this is a limitation of the ideal contact model, not a guarantee for a real road.

**Türkçe:** Formüllerin yalnızca sayısal sonucunu değil, geçerlilik koşulunu kontrol et. Negatif bir alt $v^2$ değeri “bu modelde pozitif alt sınır yok” anlamına gelebilir. Bunlar ideal temas modelinin sonuçlarıdır, gerçek araç için güvenlik garantisi değildir.

### Interactive: Banked Curve Calculator

Adjust the bank angle, curve radius, and vehicle speed. The diagram shows the forces and tells you whether the vehicle will slide, stay, or need friction.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def banked_curve(theta_deg=20.0, radius=50.0, speed=15.0, mu=0.3):
    """Required force balance and feasible speed interval in the ideal contact model."""
    if radius <= 0 or speed < 0 or mu < 0 or not 0 <= theta_deg < 90:
        raise ValueError('Use positive radius, nonnegative speed/friction, and 0 <= angle < 90°.')
    g = 9.81
    theta = np.radians(theta_deg)
    tangent = np.tan(theta)
    ac = speed**2/radius
    v_ideal = np.sqrt(radius*g*tangent)
    v_min = np.sqrt(max(0.0, radius*g*(tangent-mu)/(1+mu*tangent)))
    denominator = 1-mu*tangent
    v_max = np.sqrt(radius*g*(tangent+mu)/denominator) if denominator > 0 else np.inf
    # Per-unit-mass forces: positive friction is up the bank.
    normal = g*np.cos(theta)+ac*np.sin(theta)
    friction = g*np.sin(theta)-ac*np.cos(theta)
    feasible = abs(friction) <= mu*normal+1e-10
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    ax1.set(xlim=(-3,3), ylim=(-2.5,3), aspect='equal')
    along = np.array([np.cos(theta),np.sin(theta)])
    outward_normal = np.array([-np.sin(theta),np.cos(theta)])
    ends = np.array([-2.6*along,2.6*along])
    ax1.plot(ends[:,0],ends[:,1],color='black',lw=3)
    ax1.plot(0,0,'o',color='steelblue',markersize=15)
    scale = 1.7/max(g,normal,abs(friction))
    forces=[(np.array([0,-g]),'green','Weight'),
            (normal*outward_normal,'red','Normal'),
            (friction*along,'darkorange','Required friction')]
    for vector,color,label in forces:
        ax1.annotate('',xy=scale*vector,xytext=(0,0),
                     arrowprops=dict(arrowstyle='->',color=color,lw=2.7))
        ax1.plot([],[],color=color,lw=2.7,label=label)
    ax1.set_title('Required forces, rear view\nCentre is to the left; high bank is to the right')
    ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.04),ncol=1,fontsize=9)
    ax1.axis('off')
    upper = max(speed*1.3,v_ideal*1.4,v_min*1.3,40)
    if np.isfinite(v_max):upper=max(upper,v_max*1.15)
    ax2.set(xlim=(0,upper),ylim=(0,1),xlabel='Speed (m/s)')
    ax2.set_yticks([])
    ax2.axvspan(v_min,min(v_max,upper),alpha=0.15,color='green',label='Feasible interval in model')
    for value,label,color in [(v_ideal,'No-friction speed','blue'),(speed,'Selected speed','red')]:
        ax2.axvline(value,color=color,lw=2,label=f'{label}: {value:.2f}')
    if v_min > 0:ax2.axvline(v_min,color='orange',ls=':',label=f'Minimum: {v_min:.2f}')
    if np.isfinite(v_max):ax2.axvline(v_max,color='orange',ls=':',label=f'Maximum: {v_max:.2f}')
    ax2.set_title('Within friction limit' if feasible else 'Required friction exceeds the limit',color='green' if feasible else 'red')
    ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.27),fontsize=9)
    upper_label=f'{v_max:.2f} m/s' if np.isfinite(v_max) else 'no finite upper bound in this ideal model'
    fig.text(0.04,0.03,f'Feasible speeds: {v_min:.2f} m/s to {upper_label}.\n'
             f'Required N/m = {normal:.2f} N/kg; friction/m = {friction:+.2f} N/kg (up-bank positive).',fontsize=10)
    fig.tight_layout(rect=[0,0.24,1,0.94])
    plt.show()
    # Numerical values are explained in the figure; do not display a Python dictionary.
    return None

physics_interact(banked_curve,
         theta_deg=FloatSlider(min=5, max=60, step=1, value=20, description='Angle (deg)'),
         radius=FloatSlider(min=10, max=200, step=5, value=50, description='Radius (m)'),
         speed=FloatSlider(min=1, max=50, step=0.5, value=15, description='Speed (m/s)'),
         mu=FloatSlider(min=0.0, max=0.8, step=0.05, value=0.3, description='Friction'));

### Theory: Loop-the-Loop

For a point mass on the **inside** of a smooth circular track, normal force can push but cannot pull. At the top, inward is downward:

$$mg+N=\frac{mv_{\rm top}^2}{R},\qquad N=m\left(\frac{v_{\rm top}^2}{R}-g\right).$$

The contact limit $N=0$ gives $v_{\min,\rm top}=\sqrt{gR}$. Below that speed the required normal force would be negative, so the assumed circular contact is impossible.

To connect bottom and top speeds, preview conservation of mechanical energy for **no friction and no rotational kinetic energy**:

$$
\begin{aligned}
\frac12mv_b^2&=\frac12mv_t^2+mg(2R),\\
v_b^2&=v_t^2+4gR &&\text{multiply all terms by }2/m,\\
v_{\min,b}^2&=gR+4gR=5gR,\\
v_{\min,b}&=\sqrt{5gR}.
\end{aligned}
$$

**At other points:** If $\phi$ is measured from the bottom, $N/m=\frac{v^2}{R}+g\cos\phi$. Check both ends: at the bottom it gives $N/m=\frac{v^2}{R}+g$; at the top, $N/m=\frac{v^2}{R}-g$.

**Türkçe:** Merkez yönü konuma göre değişir. Altta normal yukarı, ağırlık aşağı olduğu için çıkarılır; tepede ikisi de merkeze yönelir. Çok düşük giriş hızında cisim tepeye çıkmadan yavaşlayıp geri dönebilir; her başarısız giriş “yoldan kopma” değildir.

### Interactive: Loop-the-Loop Simulator

Set the entry speed at the bottom of the loop and watch the ball travel around. Low speeds can turn back while maintaining contact; intermediate speeds lose contact. The animation stops at the next track contact because a collision model would be needed afterward. Frames illustrate positions, not equal time steps.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def loop_motion(entry_speed, R=2.0, g=9.81):
    """Return ideal inside-track positions, ending at the next unmodelled contact."""
    if entry_speed < 0 or R <= 0 or g <= 0:
        raise ValueError('Use nonnegative speed and positive radius/gravity.')
    v2 = entry_speed**2
    rows = []
    if v2 <= 2*g*R:
        angle = np.arccos(np.clip(1-v2/(2*g*R),-1,1))
        outward = np.linspace(0,angle,65)
        for phi in np.r_[outward,outward[-2::-1]]:
            speed = np.sqrt(max(0,v2-2*g*R*(1-np.cos(phi))))
            rows.append((R*np.sin(phi),R*(1-np.cos(phi)),speed,'on track'))
        status = 'Turns back while in contact' if entry_speed > 0 else 'At rest at the bottom'
    elif v2 < 5*g*R:
        angle = np.arccos((2*g*R-v2)/(3*g*R))
        for phi in np.linspace(0,angle,75):
            speed = np.sqrt(max(0,v2-2*g*R*(1-np.cos(phi))))
            rows.append((R*np.sin(phi),R*(1-np.cos(phi)),speed,'on track'))
        x0,y0,speed,_ = rows[-1]
        vx,vy = speed*np.cos(angle),speed*np.sin(angle)
        # Next intersection of the ballistic path with the same circle is t=4*vy/g.
        # Stop there: an impact law would be needed to continue correctly.
        contact_time = 4*vy/g
        for t in np.linspace(0,contact_time,60)[1:]:
            rows.append((x0+vx*t,y0+vy*t-0.5*g*t*t,
                         np.hypot(vx,vy-g*t),'free flight'))
        status = 'Loses contact; stop at next track contact (impact not modelled)'
    else:
        angle = 2*np.pi
        for phi in np.linspace(0,angle,150):
            speed=np.sqrt(max(0,v2-2*g*R*(1-np.cos(phi))))
            rows.append((R*np.sin(phi),R*(1-np.cos(phi)),speed,'on track'))
        status = 'Completes loop'
    return {'positions':rows,'status':status,'event_angle':angle,'minimum_entry_speed':np.sqrt(5*g*R)}

def loop_the_loop(entry_speed=8.0):
    """Illustrate physically allowed paths; frames are not equal time intervals."""
    R=2.0
    motion=loop_motion(entry_speed,R)
    rows=motion['positions']
    fig,ax=plt.subplots(figsize=(8,7))
    phi=np.linspace(0,2*np.pi,250)
    ax.plot(R*np.sin(phi),R*(1-np.cos(phi)),'k-',lw=2,label='Track')
    ax.set(xlim=(-3,3),ylim=(-0.5,4.6),aspect='equal',xlabel='x (m)',ylabel='y (m)')
    ax.set_title(f'Inside a smooth loop: entry speed {entry_speed:.1f} m/s')
    path=np.array([row[:2] for row in rows])
    ax.plot(path[:,0],path[:,1],color='gray',alpha=0.4,lw=1)
    ball,=ax.plot([],[],'ro',markersize=12)
    trail,=ax.plot([],[],'r-',alpha=0.6,lw=1.5)
    info=fig.text(0.08,0.04,'',fontsize=10)
    def init():
        ball.set_data([],[]);trail.set_data([],[])
        return ball,trail,info
    def update(frame):
        x,y,speed,phase=rows[frame]
        ball.set_data([x],[y]);trail.set_data(path[:frame+1,0],path[:frame+1,1])
        info.set_text(f'{motion["status"]}\nPhase: {phase}; speed: {speed:.2f} m/s; '
                      f'full-loop threshold: {motion["minimum_entry_speed"]:.2f} m/s.\n'
                      'Frames show positions, not equal time intervals; no rolling energy or friction.')
        return ball,trail,info
    fig.tight_layout(rect=[0,0.20,1,1])
    anim=FuncAnimation(fig,update,init_func=init,frames=physics_frames(len(rows)),
                       interval=physics_interval(len(rows),50),blit=False)
    plt.close(fig)
    return physics_animation_html(anim)

physics_interact(loop_the_loop,
         entry_speed=FloatSlider(min=2, max=15, step=0.5, value=8.0,
                                 description='v_entry (m/s)'));

### Theory: Conical Pendulum

A mass $m$ on a string of length $L$ traces a horizontal circle. The string makes angle $\theta$ with the vertical, so the path radius is $r=L\sin\theta$, not $L$. Use $T_{\rm tension}$ for string force and $T_{\rm period}$ for time per revolution.

**Free-body analysis:**

$$
\begin{aligned}
T_{\rm tension}\cos\theta&=mg &&\text{vertical balance},\\
T_{\rm tension}\sin\theta&=m\omega^2r &&\text{inward force},\\
\tan\theta&=\frac{\omega^2r}{g}=\frac{\omega^2L\sin\theta}{g} &&\text{divide the equations}.
\end{aligned}
$$

For a nonzero cone angle, write $\tan\theta=\frac{\sin\theta}{\cos\theta}$ and cancel the nonzero $\sin\theta$:

$$\frac1{\cos\theta}=\frac{\omega^2L}{g}\quad\Rightarrow\quad\cos\theta=\frac{g}{\omega^2L}\quad\Rightarrow\quad\omega=\sqrt{\frac{g}{L\cos\theta}}.$$

**Small example:** If $L=1.0\,\mathrm m$ and $\omega^2=2g/L$, then $\cos\theta=1/2$, so $\theta=60^\circ$ and $r=L\sin60^\circ=0.866\,\mathrm m$. A nonzero steady cone requires $\omega>\sqrt{g/L}$. At equality the cone shrinks to zero radius; below it the assumed nonzero cone has no solution.

**Türkçe:** $\sin\theta$ ile sadeleştirme, bu değer sıfır değilse yapılabilir. Dikey duran sarkaç ayrı bir sınır durumudur. Dairenin yarıçapı ip uzunluğundan küçük olan yatay izdüşümdür; düşey izdüşüm ise dönme düzleminin askı noktasından ne kadar aşağıda olduğunu verir.

#### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

**Think:** A point mass just maintains contact at the top of an inside loop with $R=2\,\mathrm m$. Find its top speed. Does zero normal force mean zero acceleration?

**Pair:** Compare the chosen signs and the equation before calculating.

**Explain:** Give the result, its unit, and one sentence of physical meaning.

**Türkçe:** Önce tek başına tahmin et; sonra arkadaşınla eksen ve denklem seçimini karşılaştır. Aşağıdaki model açıklamayı kendi gerekçenden sonra oku.

<details><summary>Model answer / Örnek yanıt — open after trying</summary>

At the top, inward is down. Set $N=0$ in the radial force equation, then cancel nonzero mass:

$$\begin{aligned}
mg+N&=m\frac{v^2}{R},\\
v^2&=gR=(9.81\,\mathrm{m/s^2})(2\,\mathrm m)=19.6\,\mathrm{m^2/s^2},\\
v&=\sqrt{19.6\,\mathrm{m^2/s^2}}=4.43\,\mathrm{m/s},\\
a_r&=\frac{v^2}{R}=g.
\end{aligned}$$

Gravity supplies the full downward net force at this instant. **Türkçe:** Normal kuvvetin sıfır olması bütün kuvvetlerin sıfır olması değildir. Ağırlık, merkeze yönelen ivmeyi tek başına sağlar.

</details>

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.


### Interactive: Conical Pendulum Simulator

Change the angular speed and string length to see how the cone angle responds.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def conical_pendulum(omega=3.0, L=1.5):
    """Animate a conical pendulum in 3D-like top and side views."""
    g = 9.81
    if omega <= 0 or L <= 0:
        print("Use positive angular speed and string length.")
        return
    cos_theta = g / (omega**2 * L)

    if cos_theta >= 1.0:
        print(f'Angular speed too low: omega must be > {np.sqrt(g/L):.2f} rad/s for this L.')
        print('The pendulum just hangs vertically.')
        return
    if cos_theta <= 0:
        print('Angular speed too high for this string length.')
        return

    theta = np.arccos(cos_theta)
    r = L * np.sin(theta)
    h = L * np.cos(theta)
    T_tension = g / cos_theta  # T/m
    period = 2 * np.pi / omega

    n_frames = 80
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))

    # Side view
    extent = max(0.6, 1.2*L)
    ax1.set_xlim(-extent, extent)
    ax1.set_ylim(-1.2*L, 0.25*L)
    ax1.set_aspect('equal')
    ax1.set_title('Side View', fontsize=13)
    ax1.set_xlabel('x (m)')
    ax1.set_ylabel('y (m)')
    ax1.plot(0, 0, 'ks', markersize=10)  # pivot

    # Draw circular path (dashed)
    circle_x = np.linspace(-r, r, 100)
    ax1.plot(circle_x, -h * np.ones_like(circle_x), 'b--', alpha=0.2)

    string_line, = ax1.plot([], [], 'k-', lw=2)
    ball1, = ax1.plot([], [], 'ro', markersize=14)
    info1 = fig.text(0.05, 0.025, '', fontsize=9,
                     bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    # Top view
    ax2.set_xlim(-extent, extent)
    ax2.set_ylim(-extent, extent)
    ax2.set_aspect('equal')
    ax2.set_title('Top View', fontsize=13)
    ax2.set_xlabel('x (m)')
    ax2.set_ylabel('z (m)')
    circ = plt.Circle((0, 0), r, fill=False, ls='--', color='gray', alpha=0.5)
    ax2.add_patch(circ)
    ax2.plot(0, 0, 'ks', markersize=8)
    ball2, = ax2.plot([], [], 'ro', markersize=14)
    radius_line, = ax2.plot([], [], 'k--', alpha=0.3)

    def init():
        return ball1, string_line, ball2, radius_line

    def update(frame):
        phi = 2 * np.pi * frame / n_frames
        bx = r * np.cos(phi)
        bz = r * np.sin(phi)
        by = -h

        # Side view (project onto x-y plane)
        string_line.set_data([0, bx], [0, by])
        ball1.set_data([bx], [by])

        info1.set_text(
            f'theta = {np.degrees(theta):.1f} deg; r = {r:.2f} m; '
            f'omega = {omega:.1f} rad/s; period = {period:.2f} s\n'
            f'Tension/m = {T_tension:.1f} N/kg'
        )

        # Top view
        ball2.set_data([bx], [bz])
        radius_line.set_data([0, bx], [0, bz])

        return ball1, string_line, ball2, radius_line

    fig.tight_layout(rect=[0, 0.12, 1, 1])
    anim = FuncAnimation(fig, update, init_func=init,
                         frames=physics_frames(n_frames), interval=physics_interval(n_frames, 50), blit=False)
    plt.close(fig)
    return physics_animation_html(anim)

physics_interact(conical_pendulum,
         omega=FloatSlider(min=2.0, max=8.0, step=0.2, value=3.0, description='omega (rad/s)'),
         L=FloatSlider(min=0.5, max=3.0, step=0.1, value=1.5, description='L (m)'));

### Worked Examples

#### Example 1 -- Car on a flat curve

A $1200\,\mathrm{kg}$ car rounds a flat curve of radius $80\,\mathrm m$ at $25\,\mathrm{m/s}$. What friction force is required?

#### Hand calculation: identify the force that makes the turn

Choose inward horizontal positive. Vertical acceleration is zero, so normal force equals weight. Static friction supplies the radial net force:

$$\begin{aligned}
N&=mg=(1200)(9.81)=11772\,\mathrm N,\\
a_r&=\frac{v^2}{r}=\frac{(25\,\mathrm{m/s})^2}{80\,\mathrm m}=7.81\,\mathrm{m/s^2},\\
f_s&=ma_r=(1200\,\mathrm{kg})(7.81\,\mathrm{m/s^2})=9375\,\mathrm N.
\end{aligned}$$

This is the **required** friction, directed inward. Check whether the road can supply it using the static limit:

$$\begin{aligned}
f_s&\le\mu_sN,\\
\mu_s&\ge\frac{f_s}{N}=\frac{9375\,\mathrm N}{11772\,\mathrm N}=0.796.
\end{aligned}$$

Divide by the positive normal force; the inequality keeps its direction. The coefficient has no unit because the newtons cancel.

**Check:** Do not add a second “centripetal force” arrow. Friction itself supplies the inward force.

**Türkçe:** Önce gereken kuvveti hesaplıyoruz, yolun verebileceği kuvveti değil. Kaymadan dönme koşulu, gereken sürtünmenin statik sınıra eşit veya daha küçük olmasıdır. Katsayıyı bulurken iki kuvveti birbirine böldüğümüz için birimler sadeleşir.


In [ ]:
#@title Optional numerical check — the algebra is explained above
m, v, r = 1200, 25, 80
F_c = m * v**2 / r
print(f'Required friction force: {F_c:.0f} N')
print(f'Required mu_s >= F_c / (mg) = {F_c / (m*9.81):.3f}')

#### Example 2 -- Banked road

A highway curve has radius $120\,\mathrm m$ and bank angle $15^\circ$. What is its ideal no-friction speed?

#### Hand calculation: divide away an unknown normal force

Choose inward horizontal and upward vertical axes. Normal force $N$ has components along both:

$$\begin{aligned}
N\cos15^\circ&=mg &&\text{vertical balance},\\
N\sin15^\circ&=m\frac{v^2}{r} &&\text{inward acceleration}.
\end{aligned}$$

Divide the entire radial equation by the vertical equation. The nonzero normal force and mass cancel:

$$\begin{aligned}
\frac{N\sin15^\circ}{N\cos15^\circ}&=\frac{mv^2/r}{mg},\\
\tan15^\circ&=\frac{v^2}{rg},\\
v^2&=rg\tan15^\circ &&\text{multiply both sides by }rg,\\
v&=\sqrt{(120\,\mathrm m)(9.81\,\mathrm{m/s^2})(0.268)},\\
 &=17.8\,\mathrm{m/s}.
\end{aligned}$$

Take the positive root for speed. Convert after finishing the SI calculation:

$$v=(17.8\,\mathrm{m/s})\frac{3600\,\mathrm s}{1\,\mathrm h}\frac{1\,\mathrm{km}}{1000\,\mathrm m}=63.9\,\mathrm{km/h}.$$

**Check:** The ratio $v^2/(rg)$ is dimensionless and equals $\tan15^\circ$. Without friction there is one ideal speed for this bank and radius, not an interval.

**Türkçe:** Aynı kuvvetin iki bileşenini iki ayrı yönde kullanıyoruz. Denklemleri bölünce $N$ ve $m$ sadeleşir. Son karekök adımı hızın karesini hız yapar; birim dönüşümünü en sonda gerçekleştir.


In [ ]:
#@title Optional numerical check — the algebra is explained above
r, theta_deg = 120, 15
g = 9.81
theta = np.radians(theta_deg)
v_ideal = np.sqrt(r * g * np.tan(theta))
print(f'Ideal speed: {v_ideal:.2f} m/s = {v_ideal*3.6:.1f} km/h')

#### Example 3 -- Loop-the-loop

A roller coaster loop has radius $8\,\mathrm m$. What minimum speed is needed at the bottom to complete the loop?

#### Hand calculation: force at the top, energy between heights

Model the car as a point mass sliding without friction on the inside of the loop. Ignore rotation. Let $v_t$ and $v_b$ be speeds at the top and bottom.

**First: maintain contact at the top.** Inward is down, so weight and normal force add. At the contact limit, $N=0$:

$$\begin{aligned}
mg+N&=\frac{mv_t^2}{R},\\
g&=\frac{v_t^2}{R} &&\text{set }N=0\text{ and divide by }m,\\
v_t^2&=gR &&\text{multiply by }R,\\
v_t&=\sqrt{(9.81\,\mathrm{m/s^2})(8\,\mathrm m)}=8.86\,\mathrm{m/s}.
\end{aligned}$$

**Second: account for the climb.** The top is $2R$ above the bottom. Use energy conservation, a preview of the energy notes:

$$\begin{aligned}
\frac12mv_b^2&=\frac12mv_t^2+mg(2R),\\
v_b^2&=v_t^2+4gR &&\text{multiply every term by }2/m,\\
 &=gR+4gR=5gR,\\
v_b&=\sqrt{5(9.81\,\mathrm{m/s^2})(8\,\mathrm m)}=19.8\,\mathrm{m/s}.
\end{aligned}$$

**Result:** The minimum bottom speed is $19.8\,\mathrm{m/s}$.

**Check:** It is $\sqrt5$ times the minimum top speed. The car must climb while retaining enough speed to maintain contact.

**Türkçe:** Üstteki kuvvet denklemi temas için gereken üst hızı verir. Alt hıza ulaşmak için yükseklik farkını enerjiye eklemek gerekir. $2/m$ ile çarparken yalnızca ilk terimi değil, denklemin bütün terimlerini çarp.


In [ ]:
#@title Optional numerical check — the algebra is explained above
R = 8.0
g = 9.81
v_top_min = np.sqrt(g * R)
v_bot_min = np.sqrt(5 * g * R)
print(f'Min speed at top: {v_top_min:.2f} m/s')
print(f'Min speed at bottom: {v_bot_min:.2f} m/s = {v_bot_min*3.6:.1f} km/h')

<a id="x05-problems"></a>

## 3. Problem set — predict, then check / Problem seti

Reach the symbolic answer and check a limiting case before opening the **Answer**.


### Core problems (L1) / Temel problemler

Everyone should complete these; they follow the worked examples directly.

#### P1 — Centripetal Acceleration  ·  L1
A car travels around a circular track of radius $50.0\,\mathrm m$ at a constant speed of $15.0\,\mathrm{m/s}$. (a) What is the centripetal acceleration? (b) If the car's mass is $1200\,\mathrm{kg}$, what centripetal force is required?

**Türkçe — nasıl düşünmeli:** Sabit sürat ivmenin sıfır olması anlamına gelmez; yön değişmektedir. İstenen kuvvet, gerçek kuvvetlerin merkeze doğru net bileşenidir.

<details>
<summary>Answer / Kısa kontrol</summary>

(a) $a_c=\frac{v^2}{r}=225/50=4.50\,\mathrm{m/s^2}$. (b) $\sum F_r=ma_c=(1200)(4.50)=5400\,\mathrm N$ inward.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P1 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).

#### P2 — Period and Frequency  ·  L1
A centrifuge rotor spins at $3000\,\mathrm{rpm}$ with a radius of $0.15\,\mathrm m$. (a) What is the frequency in Hz? (b) What is the period? (c) What is the centripetal acceleration at the rim, expressed in multiples of $g$?

**Türkçe — nasıl düşünmeli:** Dakikadaki devir sayısını önce saniyedeki devire çevir. Frekansın tersi periyottur; ivmeyi $g$ cinsinden vermek için hesaplanan ivmeyi yerçekimi ivmesine böl.

<details>
<summary>Answer / Kısa kontrol</summary>

(a) $f=3000/60=50\,\mathrm{Hz}$. (b) $T_{\rm period}=1/f=0.020\,\mathrm s$. (c) $\omega=2\pi f=314\,\mathrm{rad/s}$; $a_c=\omega^2r=14800\,\mathrm{m/s^2}=1509.1g\approx1510g$.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P2 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


#### P3 — Flat Curve Friction  ·  L1
A $1500\,\mathrm{kg}$ car rounds a flat curve of radius $80.0\,\mathrm m$. The coefficient of static friction between the tires and road is 0.55. What is the maximum speed the car can have without sliding?

**Türkçe — nasıl düşünmeli:** Düz virajda statik sürtünme içe doğru kuvvet sağlar. Kayma sınırında gerekli kuvvet ile en büyük statik sürtünmeyi eşitle; kütlenin sadeleştiğini göster.

<details>
<summary>Answer / Kısa kontrol</summary>

$\mu_smg=m\frac{v^2}{r}$ gives $v=\sqrt{\mu_sgr}=\sqrt{(0.55)(9.81)(80)}=20.8\,\mathrm{m/s}=74.8\,\mathrm{km/h}$.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P3 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).

#### P4 — Satellite Orbital Speed  ·  L1
A satellite orbits Earth at an altitude of $400\,\mathrm{km}$ above the surface. Earth's radius is $6371\,\mathrm{km}$ and $g$ at that altitude is approximately $8.69\,\mathrm{m/s^2}$. Find the orbital speed and period.

**Türkçe — nasıl düşünmeli:** Yörünge yarıçapı yalnızca yükseklik değildir; Dünya yarıçapı ile yüksekliği topla ve metreye çevir. Verilen yerçekimi ivmesi o yükseklik içindir; dairesel yörünge varsayılır.

<details>
<summary>Answer / Kısa kontrol</summary>

Circular-orbit radius $r=6.77\times10^6\,\mathrm m$. Speed $v=\sqrt{8.69r}=7670\,\mathrm{m/s}\approx27600\,\mathrm{km/h}$. Period $T_{\rm period}=\frac{2\pi r}{v}=5550\,\mathrm s=92.4\,\mathrm{min}$.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P4 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


### Pause and explain / Dur ve açıkla

Before the intermediate problems, explain one core result to a partner.

#### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

**Think:** An ideal bank has $\tan\theta=0.25$ and radius $40\,\mathrm m$. Find its frictionless speed and explain what changing vehicle mass does.

**Pair:** Compare the chosen signs and the equation before calculating.

**Explain:** Give the result, its unit, and one sentence of physical meaning.

**Türkçe:** Önce tek başına tahmin et; sonra arkadaşınla eksen ve denklem seçimini karşılaştır. Aşağıdaki model açıklamayı kendi gerekçenden sonra oku.

<details><summary>Model answer / Örnek yanıt — open after trying</summary>

The radial and vertical equations contain the same $N$ and $m$. Divide them to eliminate both:

$$\begin{aligned}
\frac{N\sin\theta}{N\cos\theta}&=\frac{mv^2/r}{mg},\\
v^2&=rg\tan\theta=(40)(9.81)(0.25)=98.1\,\mathrm{m^2/s^2},\\
v&=\sqrt{98.1\,\mathrm{m^2/s^2}}=9.90\,\mathrm{m/s}.
\end{aligned}$$

Changing mass does not change this ideal speed. **Türkçe:** Kütlenin sadeleşmesi daha ağır aracın kuvvet gerektirmediği anlamına gelmez. Gereken kuvvet kütleyle artar, fakat bu kuvvet oranında kütleler sadeleşir.

</details>

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### Intermediate problems (L2) / Orta düzey

Combine two ideas from this week.

#### P5 — Banked Curve Design  ·  L2
A highway curve of radius $200\,\mathrm m$ is designed for traffic at $90\,\mathrm{km/h}$. (a) What bank angle is needed with no friction? (b) If the road is wet ($\mu_s=0.20$), what are the minimum and maximum safe speeds? (c) Express the maximum speed in km/h.

**Türkçe — nasıl düşünmeli:** Önce tasarım hızını SI birimine çevirip sürtünmesiz bank açısını bul. Alt ve üst hız sınırında sürtünme yönü farklıdır; ara hesaplarda açıyı erken yuvarlama.

<details>
<summary>Answer / Kısa kontrol</summary>

(a) $v_{\rm design}=25\,\mathrm{m/s}$; $\tan\theta=0.319$ and $\theta=17.7^\circ$. (b) $v_{\min}=\sqrt{rg(\tan\theta-\mu)/(1+\mu\tan\theta)}=14.8\,\mathrm{m/s}$; $v_{\max}=\sqrt{rg(\tan\theta+\mu)/(1-\mu\tan\theta)}=33.0\,\mathrm{m/s}$. (c) $v_{\max}=119\,\mathrm{km/h}$. Use the unrounded bank angle.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P5 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


#### P6 — Vertical Loop -- Normal Force  ·  L2
A $0.50\,\mathrm{kg}$ ball on a string moves in a vertical circle of radius $1.20\,\mathrm m$. At the top of the circle, the speed is $4.0\,\mathrm{m/s}$. (a) What is the tension in the string at the top? (b) Using energy conservation, find the speed at the bottom. (c) What is the tension at the bottom?

**Türkçe — nasıl düşünmeli:** Başlıkta normal kuvvet yazsa da düzenek ipli bir toptur: kuvvet gerilmedir. Tepede ve altta merkeze doğru yön değişir; iki hız arasındaki bağıntı enerji gerektirir.

<details>
<summary>Answer / Kısa kontrol</summary>

The ball is on a string, so the force is tension. (a) $T_{\rm top}=0.50(16/1.20-9.81)=1.76\,\mathrm N$. (b) $v_b^2=16+4(9.81)(1.20)=63.1\,\mathrm{m^2/s^2}$, so $v_b=7.94\,\mathrm{m/s}$. (c) $T_b=0.50(63.1/1.20+9.81)=31.2\,\mathrm N$.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P6 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


#### P7 — Conical Pendulum  ·  L2
A $0.30\,\mathrm{kg}$ mass hangs from a $0.80\,\mathrm m$ string and swings in a horizontal circle, making an angle of $25^\circ$ with the vertical. (a) Find the radius of the circular path. (b) Find the speed of the mass. (c) Find the tension in the string. (d) Find the period of revolution.

**Türkçe — nasıl düşünmeli:** Yörünge yarıçapı ip uzunluğunun yatay izdüşümüdür. Gerilme ile dönme periyodu bazen aynı harfle gösterildiğinden sembolleri açıkça etiketle.

<details>
<summary>Answer / Kısa kontrol</summary>

(a) $r=0.80\sin25^\circ=0.338\,\mathrm m$. (b) $v=\sqrt{rg\tan25^\circ}=1.24\,\mathrm{m/s}$. (c) $T_{\rm tension}=0.30g/\cos25^\circ=3.25\,\mathrm N$. (d) $T_{\rm period}=\frac{2\pi r}{v}=1.71\,\mathrm s$. Tension and period are different quantities.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P7 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


#### P8 — Car Over a Hill  ·  L2
A $1000\,\mathrm{kg}$ car drives over the top of a hill that can be approximated as a circular arc of radius $40.0\,\mathrm m$. (a) At what speed does the car begin to lose contact with the road ($N=0$)? (b) If the car goes over the hill at $15.0\,\mathrm{m/s}$, what is the normal force on the car? (c) What does the driver feel in terms of apparent weight?

**Türkçe — nasıl düşünmeli:** Tepe üzerinde merkez aşağıdadır, normal kuvvet yukarıdadır. Otomobile etki eden normal kuvveti sürücünün tartı kuvveti gibi yorumlama; sürücünün kütlesi verilmemiştir.

<details>
<summary>Answer / Kısa kontrol</summary>

(a) Contact limit $v=\sqrt{gr}=19.8\,\mathrm{m/s}$. (b) Normal force on the car $N=m(g-\frac{v^2}{r})=4185\,\mathrm N$. (c) The driver feels lighter: apparent weight is $42.7\%$ of their own weight if they follow the same path. The driver’s mass is not supplied; $4185\,\mathrm N$ is not their contact force.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P8 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


### Challenge problems (L3) / İleri düzey

Engineering-style problems with several steps; useful preparation for the exams.

#### P9 — Loop-the-Loop with Friction  ·  L3
A small block slides from rest down a frictionless ramp of height $h$ and enters a vertical loop of radius $R=1.50\,\mathrm m$. The loop has a coefficient of kinetic friction $\mu_k=0.10$. (a) Find the minimum height $h$ such that the block barely maintains contact at the top of the loop. (Hint: at the top, use both Newton's 2nd law and energy conservation with friction loss.) (b) Compare this to the frictionless case.

**Türkçe — nasıl düşünmeli:** Bu ileri düzey soru enerji kaybını içerir. Döngü boyunca normal kuvvet değiştiği için sürtünme işini tek bir sabit kuvvet çarpı tam çevre kabul etme; önce kuvvet ve enerji denklemlerini kur.

<details>
<summary>Answer / Kısa kontrol</summary>

For a block sliding inside the ideal circular loop, (a) $h_{\min}=5.47\,\mathrm m$. (b) Frictionless $h=2.5R=3.75\,\mathrm m$; increase $1.72\,\mathrm m$ or $45.8\%$. Use $W_f=\int_0^\pi\mu_kNR\,d\theta$ over the bottom-to-top half-loop, with $N/m=\frac{v^2}{R}+g\cos\theta$. The normal force varies; approximating it by $mg$ and using the full circumference is incorrect. The complete calculation requires an advanced integral; first focus on force and energy setup.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P9 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


#### P10 — Banked Curve Design for an Autonomous Vehicle  ·  L3
You are designing a banked test track for autonomous vehicles. The curve has radius $150\,\mathrm m$. Vehicles will travel between $40\,\mathrm{km/h}$ and $120\,\mathrm{km/h}$. The tire-road friction coefficient is $\mu_s=0.40$. (a) Find the bank angle that requires no friction at $80\,\mathrm{km/h}$ (the median speed). (b) Verify that vehicles at $40\,\mathrm{km/h}$ and $120\,\mathrm{km/h}$ can navigate the curve without sliding. (c) What is the absolute maximum speed before sliding occurs? (d) At maximum speed, what is the centripetal acceleration as a multiple of $g$?

**Türkçe — nasıl düşünmeli:** Banka açısı tasarım hızından bulunur; sonra her uç hız için sürtünme gereksinimini kontrol et. Negatif bir cebirsel alt hız karesi, fiziksel alt sınırın sıfır olabileceğini gösterir.

<details>
<summary>Answer / Kısa kontrol</summary>

(a) $\theta=18.6^\circ$ with $\tan\theta=0.336$. (b) Physical speed interval: $0$–$35.4\,\mathrm{m/s}$, so both $40$ and $120\,\mathrm{km/h}$ satisfy the ideal model. Negative algebraic lower $v^2$ means no positive minimum speed is needed. (c) $v_{\max}=35.4\,\mathrm{m/s}=127\,\mathrm{km/h}$. (d) $a_c=8.34\,\mathrm{m/s^2}=0.84965g$.

</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 05 P10 in the solutions collection (file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026).


## Solutions / Çözümler

Complete worked solutions: Module 05 (Circular motion), file `Week_05_Python_Solutions.ipynb`, opens 18 December 2026 on the course page.

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)